In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import perf_counter
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
)
from sklearn.svm import SVC
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from IPython.display import display
import pickle

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None

In [ ]:
# load data
projectDir = Path.cwd().resolve().parent
dataPath = projectDir / "data" / "processed" / "matchupDiff_week5_features.csv"

matchupDiff = pd.read_csv(dataPath)
matchupDiff["y"] = matchupDiff["y"].astype(int)

In [ ]:
# dropping any columns that could cause data leakage
leakCols = [
    "WScore", "LScore", "WTeamID", "LTeamID", "winnerTeamId",
    "team1Name", "team2Name", "Unnamed: 0"
]

idCols = ["team1Id", "team2Id"]

# all dropped columns
dropCols = ["Season", "y"] + leakCols + idCols

In [ ]:
# train = 2003 - 2021, validation 2022, test 2023 - 2025
trainDf = matchupDiff[matchupDiff["Season"] <= 2021].copy()
valDf = matchupDiff[matchupDiff["Season"] == 2022].copy()
testDf = matchupDiff[matchupDiff["Season"].between(2023, 2025)].copy()

In [ ]:
def buildXy(splitDf, featureCols=None):
    x = splitDf.drop(columns=dropCols, errors="ignore")
    x = x.select_dtypes(include=[np.number])

    if featureCols is None:
        featureCols = x.columns.tolist()
    else:
        x = x.reindex(columns=featureCols)

    y = splitDf["y"].astype(int)
    return x, y, featureCols

xTrain, yTrain, featureCols = buildXy(trainDf)
xVal, yVal, _ = buildXy(valDf, featureCols)
xTest, yTest, _ = buildXy(testDf, featureCols)

print(f"Train shape: {xTrain.shape}, Val shape: {xVal.shape}, Test shape: {xTest.shape}")
print(f"Positive class rate in train: {yTrain.mean():.3f}")

In [ ]:
thresholds = [0.45, 0.50, 0.55]
randomSeed = 226


def scorePredictions(y, proba, threshold):
    preds = (proba >= threshold).astype(int)
    metrics = {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0)
    }
    cm = confusion_matrix(y, preds)
    cmDf = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])
    return metrics, cmDf


def buildModel(cfg):
    if cfg["modelFamily"] == "xgboost":
        return XGBClassifier(
            n_estimators=cfg["nEstimators"],
            max_depth=cfg["maxDepth"],
            learning_rate=cfg["learningRate"],
            subsample=cfg["subsample"],
            colsample_bytree=cfg["colsampleBytree"],
            min_child_weight=cfg["minChildWeight"],
            gamma=cfg["gamma"],
            reg_lambda=cfg["regLambda"],
            reg_alpha=cfg["regAlpha"],
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=cfg["randomState"],
            n_jobs=-1,
            tree_method="hist"
        )

    if cfg["modelFamily"] == "catboost":
        return CatBoostClassifier(
            iterations=cfg["iterations"],
            depth=cfg["depth"],
            learning_rate=cfg["learningRate"],
            l2_leaf_reg=cfg["l2LeafReg"],
            random_strength=cfg["randomStrength"],
            bagging_temperature=cfg["baggingTemperature"],
            border_count=cfg["borderCount"],
            loss_function="Logloss",
            eval_metric="Accuracy",
            verbose=0,
            allow_writing_files=False,
            random_seed=cfg["randomState"],
            thread_count=-1
        )

    if cfg["modelFamily"] == "histGradientBoosting":
        return HistGradientBoostingClassifier(
            learning_rate=cfg["learningRate"],
            max_depth=cfg["maxDepth"],
            max_leaf_nodes=cfg["maxLeafNodes"],
            min_samples_leaf=cfg["minSamplesLeaf"],
            l2_regularization=cfg["l2Regularization"],
            max_iter=cfg["maxIter"],
            random_state=cfg["randomState"]
        )

    if cfg["modelFamily"] == "extraTrees":
        return ExtraTreesClassifier(
            n_estimators=cfg["nEstimators"],
            max_depth=cfg["maxDepth"],
            min_samples_split=cfg["minSamplesSplit"],
            min_samples_leaf=cfg["minSamplesLeaf"],
            max_features=cfg["maxFeatures"],
            bootstrap=cfg["bootstrap"],
            class_weight=cfg["classWeight"],
            random_state=cfg["randomState"],
            n_jobs=-1
        )

    if cfg["modelFamily"] == "randomForest":
        return RandomForestClassifier(
            n_estimators=cfg["nEstimators"],
            max_depth=cfg["maxDepth"],
            min_samples_split=cfg["minSamplesSplit"],
            min_samples_leaf=cfg["minSamplesLeaf"],
            max_features=cfg["maxFeatures"],
            bootstrap=cfg["bootstrap"],
            class_weight=cfg["classWeight"],
            random_state=cfg["randomState"],
            n_jobs=-1
        )

    if cfg["modelFamily"] == "gradientBoosting":
        return GradientBoostingClassifier(
            n_estimators=cfg["nEstimators"],
            learning_rate=cfg["learningRate"],
            subsample=cfg["subsample"],
            min_samples_split=cfg["minSamplesSplit"],
            min_samples_leaf=cfg["minSamplesLeaf"],
            max_depth=cfg["maxDepth"],
            random_state=cfg["randomState"]
        )

    if cfg["modelFamily"] == "adaBoost":
        return AdaBoostClassifier(
            n_estimators=cfg["nEstimators"],
            learning_rate=cfg["learningRate"],
            random_state=cfg["randomState"]
        )

    if cfg["modelFamily"] == "mlp":
        return MLPClassifier(
            hidden_layer_sizes=cfg["hiddenLayerSizes"],
            alpha=cfg["alpha"],
            learning_rate_init=cfg["learningRateInit"],
            max_iter=cfg["maxIter"],
            early_stopping=cfg["earlyStopping"],
            random_state=cfg["randomState"]
        )

    if cfg["modelFamily"] == "svc":
        return SVC(
            C=cfg["cValue"],
            gamma=cfg["gamma"],
            kernel=cfg["kernel"],
            probability=True,
            random_state=cfg["randomState"]
        )

    raise ValueError(f"Unsupported model family: {cfg['modelFamily']}")


def buildPipeline(cfg):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if cfg["scaleFeatures"]:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", buildModel(cfg)))
    return Pipeline(steps=steps)


def displayConfig(cfg):
    return {k: v for k, v in cfg.items() if k not in ["scaleFeatures"]}


def normalizeValue(value):
    if isinstance(value, dict):
        return tuple((k, normalizeValue(v)) for k, v in sorted(value.items()))
    if isinstance(value, list):
        return tuple(normalizeValue(v) for v in value)
    if isinstance(value, tuple):
        return tuple(normalizeValue(v) for v in value)
    return value


def configSignature(cfg):
    signature = []
    for key, value in sorted(cfg.items()):
        signature.append((key, normalizeValue(value)))
    return tuple(signature)


def gridSize(grid):
    size = 1
    for values in grid.values():
        size *= len(values)
    return size


def sampleConfigs(modelFamily, grid, nIter, scaleFeatures, seedOffset):
    actualIter = min(nIter, gridSize(grid))
    sampled = list(ParameterSampler(grid, n_iter=actualIter, random_state=randomSeed + seedOffset))
    return [
        {
            "modelFamily": modelFamily,
            "scaleFeatures": scaleFeatures,
            "randomState": randomSeed,
            **cfg,
        }
        for cfg in sampled
    ]

In [ ]:
# broad hyperparameter search across many model families
configs = []
seenConfigs = set()

searchPlan = []

if XGBClassifier is not None:
    xgbGrid = {
        "nEstimators": [200, 400, 600, 800, 1000],
        "maxDepth": [3, 4, 5, 6, 8],
        "learningRate": [0.01, 0.03, 0.05, 0.08, 0.10],
        "subsample": [0.7, 0.85, 1.0],
        "colsampleBytree": [0.6, 0.8, 1.0],
        "minChildWeight": [1, 3, 5, 7],
        "gamma": [0.0, 0.1, 0.3],
        "regLambda": [1, 3, 5, 7],
        "regAlpha": [0.0, 0.1, 0.5]
    }
    searchPlan.append(("xgboost", xgbGrid, 80, False, 11))

if CatBoostClassifier is not None:
    catGrid = {
        "iterations": [300, 500, 700, 900, 1200],
        "depth": [4, 5, 6, 8, 10],
        "learningRate": [0.01, 0.03, 0.05, 0.08, 0.10],
        "l2LeafReg": [1, 3, 5, 7, 9],
        "randomStrength": [0.5, 1.0, 2.0],
        "baggingTemperature": [0.0, 1.0, 3.0],
        "borderCount": [64, 128, 254]
    }
    searchPlan.append(("catboost", catGrid, 80, False, 29))

histGrid = {
    "learningRate": [0.01, 0.03, 0.05, 0.08, 0.10],
    "maxDepth": [None, 4, 6, 8, 10],
    "maxLeafNodes": [15, 31, 63, 127],
    "minSamplesLeaf": [5, 10, 20, 30],
    "l2Regularization": [0.0, 0.1, 0.3, 1.0],
    "maxIter": [200, 400, 600, 800]
}
searchPlan.append(("histGradientBoosting", histGrid, 60, False, 43))

extraTreesGrid = {
    "nEstimators": [300, 500, 800, 1000],
    "maxDepth": [None, 10, 20, 30],
    "minSamplesSplit": [2, 5, 10],
    "minSamplesLeaf": [1, 2, 4],
    "maxFeatures": ["sqrt", "log2", 0.5, 0.8],
    "bootstrap": [False, True],
    "classWeight": [None, "balanced", {0: 1, 1: 1.25}, {0: 1, 1: 1.5}]
}
searchPlan.append(("extraTrees", extraTreesGrid, 50, False, 59))

randomForestGrid = {
    "nEstimators": [300, 500, 800, 1000],
    "maxDepth": [None, 10, 20, 30],
    "minSamplesSplit": [2, 5, 10],
    "minSamplesLeaf": [1, 2, 4],
    "maxFeatures": ["sqrt", "log2", 0.5, 0.8],
    "bootstrap": [False, True],
    "classWeight": [None, "balanced", {0: 1, 1: 1.25}, {0: 1, 1: 1.5}]
}
searchPlan.append(("randomForest", randomForestGrid, 50, False, 71))

gradientBoostingGrid = {
    "nEstimators": [150, 250, 400, 600],
    "learningRate": [0.01, 0.03, 0.05, 0.08, 0.10],
    "subsample": [0.7, 0.85, 1.0],
    "minSamplesSplit": [2, 5, 10],
    "minSamplesLeaf": [1, 2, 4],
    "maxDepth": [2, 3, 4, 5]
}
searchPlan.append(("gradientBoosting", gradientBoostingGrid, 40, False, 83))

adaBoostGrid = {
    "nEstimators": [100, 200, 400, 600, 800],
    "learningRate": [0.01, 0.03, 0.05, 0.08, 0.10, 0.20, 0.50, 1.0]
}
searchPlan.append(("adaBoost", adaBoostGrid, 32, False, 97))

mlpGrid = {
    "hiddenLayerSizes": [(64, 32), (96, 48), (128, 64), (128, 64, 32), (256, 128, 64)],
    "alpha": [0.00005, 0.0001, 0.0005, 0.001, 0.005],
    "learningRateInit": [0.0001, 0.0003, 0.0005, 0.001, 0.003],
    "maxIter": [1200, 1800, 2500],
    "earlyStopping": [True]
}
searchPlan.append(("mlp", mlpGrid, 40, True, 109))

svcGrid = {
    "cValue": [0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    "gamma": ["scale", 0.001, 0.01, 0.05, 0.10],
    "kernel": ["rbf", "poly", "sigmoid"]
}
searchPlan.append(("svc", svcGrid, 32, True, 127))

for modelFamily, grid, nIter, scaleFeatures, seedOffset in searchPlan:
    for cfg in sampleConfigs(modelFamily, grid, nIter, scaleFeatures, seedOffset):
        signature = configSignature(cfg)
        if signature not in seenConfigs:
            seenConfigs.add(signature)
            configs.append(cfg)

searchPlanDf = pd.DataFrame(
    [
        {
            "modelFamily": modelFamily,
            "sampledConfigs": min(nIter, gridSize(grid)),
            "rawGridSize": gridSize(grid)
        }
        for modelFamily, grid, nIter, scaleFeatures, seedOffset in searchPlan
    ]
)

print(f"Total sampled model fits: {len(configs)}")
print(f"Thresholds tested per fit: {thresholds}")
display(searchPlanDf)

In [ ]:
results = []
failedConfigs = []
configLookup = {}
overallStart = perf_counter()

for idx, cfg in enumerate(configs):
    configLookup[idx] = cfg

    try:
        pipe = buildPipeline(cfg)
        fitStart = perf_counter()
        pipe.fit(xTrain, yTrain)
        fitSeconds = perf_counter() - fitStart

        trainProba = pipe.predict_proba(xTrain)[:, 1]
        valProba = pipe.predict_proba(xVal)[:, 1]

        for threshold in thresholds:
            trainMetrics, _ = scorePredictions(yTrain, trainProba, threshold)
            valMetrics, _ = scorePredictions(yVal, valProba, threshold)

            results.append({
                "configId": idx,
                "modelFamily": cfg["modelFamily"],
                "threshold": threshold,
                "fitSeconds": fitSeconds,
                "model": str(displayConfig(cfg)),
                "set": "train",
                **trainMetrics
            })
            results.append({
                "configId": idx,
                "modelFamily": cfg["modelFamily"],
                "threshold": threshold,
                "fitSeconds": fitSeconds,
                "model": str(displayConfig(cfg)),
                "set": "val",
                **valMetrics
            })
        if (idx + 1) % 20 == 0:
            elapsedMinutes = (perf_counter() - overallStart) / 60
            print(f"Completed {idx + 1}/{len(configs)} fits in {elapsedMinutes:.1f} minutes", flush=True)
    except Exception as err:
        failedConfigs.append({
            "configId": idx,
            "modelFamily": cfg["modelFamily"],
            "model": str(displayConfig(cfg)),
            "error": str(err)
        })

resultsDf = pd.DataFrame(results)
resultsDf = resultsDf.sort_values(
    ["set", "accuracy", "recall", "precision"],
    ascending=[True, False, False, False]
)

if failedConfigs:
    display(pd.DataFrame(failedConfigs))

print(f"Successful fits: {resultsDf['configId'].nunique()}")
print(f"Failed fits: {len(failedConfigs)}")
print(f"Total elapsed minutes: {(perf_counter() - overallStart) / 60:.1f}")
resultsDf.head(12)

In [ ]:
# family level summary and top validation candidates
valOnly = resultsDf[resultsDf["set"] == "val"].copy()
valOnly = valOnly.sort_values(["accuracy", "recall", "precision"], ascending=False)

familySummary = valOnly.groupby("modelFamily").agg(
    bestAccuracy=("accuracy", "max"),
    avgAccuracy=("accuracy", "mean"),
    bestRecall=("recall", "max"),
    avgFitSeconds=("fitSeconds", "mean"),
    testedThresholds=("threshold", "nunique"),
    testedConfigs=("configId", "nunique")
).sort_values(["bestAccuracy", "bestRecall"], ascending=False)

display(familySummary)
display(valOnly.head(10))

In [ ]:
# top 3 by validation accuracy
top3Ids = valOnly.head(3)["configId"].tolist()

top3Results = resultsDf[resultsDf["configId"].isin(top3Ids)].copy()
top3Results = top3Results.sort_values(["configId", "set", "threshold"])
display(top3Results)

top3ValCompare = top3Results[top3Results["set"] == "val"].copy()
top3ValCompare = top3ValCompare.sort_values(["accuracy", "recall", "precision"], ascending=False)
display(top3ValCompare)

In [ ]:
# pick the winner on validation
bestRow = valOnly.iloc[0]
bestCfg = configLookup[int(bestRow["configId"])].copy()
bestCfg["threshold"] = float(bestRow["threshold"])

print("Best model (by validation accuracy, then recall, precision):")
print(bestRow)
print(displayConfig(bestCfg))

In [ ]:
# fit the best configuration on train and score train/validation
bestPipe = buildPipeline(bestCfg)
bestPipe.fit(xTrain, yTrain)

trainProba = bestPipe.predict_proba(xTrain)[:, 1]
valProba = bestPipe.predict_proba(xVal)[:, 1]

trainMetrics, trainCm = scorePredictions(yTrain, trainProba, bestCfg["threshold"])
valMetrics, valCm = scorePredictions(yVal, valProba, bestCfg["threshold"])

print("Train metrics:", trainMetrics)
display(trainCm)

print("Validation metrics:", valMetrics)
display(valCm)

In [ ]:
# use best model on test
trainValDf = matchupDiff[matchupDiff["Season"] <= 2022].copy()
xTrainVal, yTrainVal, _ = buildXy(trainValDf, featureCols)

bestPipe.fit(xTrainVal, yTrainVal)

testProba = bestPipe.predict_proba(xTest)[:, 1]
testMetrics, testCm = scorePredictions(yTest, testProba, bestCfg["threshold"])
print("Test metrics (2023-2025):", testMetrics)
display(testCm)

In [ ]:
# exporting model to pickle
modelDir = projectDir / "models"
modelDir.mkdir(parents=True, exist_ok=True)

modelPath = modelDir / "week8_best_model.pkl"
payload = {
    "model": bestPipe,
    "modelFamily": bestCfg["modelFamily"],
    "threshold": bestCfg["threshold"],
    "featureCols": featureCols,
    "params": displayConfig(bestCfg),
    "validationMetrics": valMetrics,
    "testMetrics": testMetrics
}

with open(modelPath, "wb") as f:
    pickle.dump(payload, f)

In [ ]:
# save model search results
processedDir = projectDir / "data" / "processed"
processedDir.mkdir(parents=True, exist_ok=True)

top3CsvPath = processedDir / "week8_top3_models.csv"
allResultsCsvPath = processedDir / "week8_all_model_results.csv"
familySummaryCsvPath = processedDir / "week8_family_summary.csv"

top3Results.to_csv(top3CsvPath, index=False)
resultsDf.to_csv(allResultsCsvPath, index=False)
familySummary.reset_index().to_csv(familySummaryCsvPath, index=False)

print(f"Saved model to: {modelPath}")
print(f"Saved top 3 results to: {top3CsvPath}")
print(f"Saved all model results to: {allResultsCsvPath}")
print(f"Saved family summary to: {familySummaryCsvPath}")